# 图片数据预处理

In [ ]:
from dataclasses import dataclass
import numpy as np

@dataclass
class Preprocessing:
    width: int
    height: int
    channels: int
    mean: tuple[float] = (0,)
    std: tuple[float] = (1,)
    layout: str = "HWC"
    name: str = "data"
    format: str = "RGB"

    def __post_init__(self):
        if self.layout == "HWC":
            self.shape = self.height, self.width, self.channels
        elif self.layout == "CHW":
            self.shape = self.channels, self.height, self.width
        else:
            raise ValueError(f"Unknown layout: {self.layout}")

    def load(self, path: str | Path) -> np.ndarray:
        """加载图片"""
        img = Image.open(path).resize((self.width, self.height)) # uint8 数据
        if self.format == "GRAY":
            img = img.convert("L")
            img = np.expand_dims(img, axis=-1) # WH->HWC
        elif self.format == "RGB":
            img = np.array(img.convert("RGB")) # WHC->HWC
        elif self.format == "BGR":
            img = np.array(img.convert("RGB")) # WHC->HWC
            img = img[..., ::-1] # RGB 转 BGR
        else:
            raise TypeError(f'暂未支持数据布局 {self.format}')
        return img
    
    def __call__(self, path: str | Path) -> np.ndarray:
        img = self.load(path)/255.0 # 归一化（将 uint8 数据归一化到 [0, 1]，这是神经网络的标准输入格式）
        img = (img - self.mean) / self.std # 标准化，使数据分布更接近标准正态分布
        img = img.astype("float32")
        if self.layout == "CHW":
            img = img.transpose(2, 0, 1) # HWC->CHW
        return img

    def torch_call(self, path: str | Path) -> torch.Tensor:
        assert self.layout == "CHW", "torchvision 只支持 CHW 布局"
        from torchvision.transforms import v2
        import torch
        from torch import nn
        inp = self.load(path)
        return nn.Sequential(
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(self.mean, self.std)
        )(inp)

In [ ]:
from pathlib import Path
from PIL import Image
root_dir = Path('../../images')
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
layout = "CHW"
preprocessing = Preprocessing(32, 32, 3, mean, std, layout)
# torch_inp = preprocessing.torch_call(root_dir/"Giant_Panda_in_Beijing_Zoo_1.jpg")
inp = preprocessing(root_dir/"Giant_Panda_in_Beijing_Zoo_1.jpg")
# np.testing.assert_almost_equal(inp, torch_inp.numpy(), decimal=6)

In [1]:
from manim import *

In [2]:
%%manim -qm ThreeDCube
from manim import *

class ThreeDCube(ThreeDScene):
    def construct(self):
        # 设置3D摄像机视角
        self.set_camera_orientation(phi=75*DEGREES, theta=-45*DEGREES)
        
        # 创建3D坐标轴（可选）
        axes = ThreeDAxes()
        
        # 创建长方体并设置样式
        cube = Cube(
            side_length=3,          # 基础边长
            fill_opacity=0.7,       # 填充透明度
            fill_color=BLUE,        # 填充颜色
            stroke_width=2,         # 边框粗细
            stroke_color=WHITE      # 边框颜色
        )
        
        # 缩放为长方体（长3，宽2，高4）
        cube.stretch_to_fit_depth(4)
        cube.stretch_to_fit_width(3)
        cube.stretch_to_fit_height(2)
        
        # 添加动画
        self.play(Create(cube))
        self.begin_ambient_camera_rotation(rate=0.2)  # 自动旋转摄像机
        self.wait(5)

Manim Community v0.18.1

[03/04/25 13:26:47] INFO     Animation 0 : Partial movie file written in                   ]8;id=786802;file:///media/pc/data/lxw/envs/anaconda3a/envs/ai/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=185317;file:///media/pc/data/lxw/envs/anaconda3a/envs/ai/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#527\527]8;;\
                             '/media/pc/data/lxw/ai/tvm-book/doc/tutorials/cv/media/videos                         
                             /cv/720p30/partial_movie_files/ThreeDCube/2450691933_80474953                         
                             3_223132457.mp4'                                                                      

# 2025-03-04 13:26:47,104 INFO manim scene_file_writer.py:527 -- Animation 0 : Partial movie file written in '/media/pc/data/lxw/ai/tvm-book/doc/tutorials/cv/media/videos/cv/720p30/partial_movie_files/ThreeDCube/2450691933_804749533_223132457.mp4'



[03/04/25 13:26:50] INFO     Animation 1 : Partial movie file written in                   ]8;id=965808;file:///media/pc/data/lxw/envs/anaconda3a/envs/ai/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=781596;file:///media/pc/data/lxw/envs/anaconda3a/envs/ai/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#527\527]8;;\
                             '/media/pc/data/lxw/ai/tvm-book/doc/tutorials/cv/media/videos                         
                             /cv/720p30/partial_movie_files/ThreeDCube/3625502138_73029631                         
                             0_2954433035.mp4'                                                                     

# 2025-03-04 13:26:50,607 INFO manim scene_file_writer.py:527 -- Animation 1 : Partial movie file written in '/media/pc/data/lxw/ai/tvm-book/doc/tutorials/cv/media/videos/cv/720p30/partial_movie_files/ThreeDCube/3625502138_730296310_2954433035.mp4'



...

# 2025-03-04 13:26:50,613 INFO manim scene_file_writer.py:617 -- Combining to Movie file.



                    INFO                                                                   ]8;id=700492;file:///media/pc/data/lxw/envs/anaconda3a/envs/ai/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=635875;file:///media/pc/data/lxw/envs/anaconda3a/envs/ai/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#737\737]8;;\
                             File ready at                                                                         
                             '/media/pc/data/lxw/ai/tvm-book/doc/tutorials/cv/media/videos                         
                             /cv/720p30/ThreeDCube.mp4'                                                            
                                                                                                                   

# 2025-03-04 13:26:50,820 INFO manim scene_file_writer.py:737 -- 
File ready at '/media/pc/data/lxw/ai/tvm-book/doc/tutorials/cv/media/videos/cv/720p30/ThreeDCube.mp4'




                    INFO     Rendered ThreeDCube                                                       ]8;id=39859;file:///media/pc/data/lxw/envs/anaconda3a/envs/ai/lib/python3.12/site-packages/manim/scene/scene.py\scene.py]8;;\:]8;id=5562;file:///media/pc/data/lxw/envs/anaconda3a/envs/ai/lib/python3.12/site-packages/manim/scene/scene.py#247\247]8;;\
                             Played 2 animations                                                                   

# 2025-03-04 13:26:50,828 INFO manim scene.py:247 -- Rendered ThreeDCube
Played 2 animations

